In [1]:
import os
import io
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import soundfile as sf
import librosa

from boxsdk import OAuth2, Client

# ML + DL
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ============================================================
# CONFIG
# ============================================================

CLIENT_ID = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
DEVELOPER_TOKEN = "YOUR_DEVELOPER_TOKEN"

SHARED_FOLDER_ID = "AI-AFS/Incoming_Data"   # from your link
ETHOGRAM_XLSX = "001128_AIAFS_datasheet_ethogram_2025_VE.xlsx"

LOCAL_AUDIO_DIR = "downloaded_wavs"
os.makedirs(LOCAL_AUDIO_DIR, exist_ok=True)

SAMPLE_RATE = 48000
WIN_SEC = 2.0
HOP_SEC = 1.0
N_MELS = 64
N_MFCC = 20

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ============================================================
# 1. CONNECT TO BOX
# ============================================================

oauth = OAuth2(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    access_token=DEVELOPER_TOKEN
)
client = Client(oauth)

# ============================================================
# 2. LIST ALL WAV FILES IN SHARED FOLDER (RECURSIVE)
# ============================================================

def list_files_recursive(folder_id):
    items = client.folder(folder_id).get_items(limit=1000)
    wav_files = []
    for item in items:
        if item.type == "file" and item.name.lower().endswith(".wav"):
            wav_files.append(item)
        elif item.type == "folder":
            wav_files.extend(list_files_recursive(item.id))
    return wav_files

wav_items = list_files_recursive(SHARED_FOLDER_ID)
print("Found WAV files:", len(wav_items))

# ============================================================
# 3. DOWNLOAD WAV FILES
# ============================================================

def download_wav(item):
    out_path = os.path.join(LOCAL_AUDIO_DIR, item.name)
    if not os.path.exists(out_path):
        with open(out_path, "wb") as f:
            client.file(item.id).download_to(f)
    return out_path

local_wavs = [download_wav(item) for item in wav_items]

# ============================================================
# 4. LOAD ETHOGRAM + LABEL MAPPING
# ============================================================

eth = pd.read_excel(ETHOGRAM_XLSX)
eth.columns = [c.strip().lower().replace(" ", "_") for c in eth.columns]

def parse_dt(date_val, time_str):
    d = pd.to_datetime(date_val).strftime("%Y-%m-%d")
    return datetime.strptime(d + " " + time_str, "%Y-%m-%d %H:%M:%S")

eth["start_dt"] = eth.apply(lambda r: parse_dt(r["date"], str(r["time_entered"])), axis=1)
eth["end_dt"]   = eth.apply(lambda r: parse_dt(r["date"], str(r["exit_time"])), axis=1)

FEED = ["feeding", "feed", "nourriss", "alimentation"]
STRESS = ["cleaning", "flush", "maintenance", "pump", "dead", "missing", "stress"]

def classify_row(row):
    text = " ".join(str(row.get(c, "")).lower() for c in eth.columns)
    if any(k in text for k in FEED): return "feeding"
    if any(k in text for k in STRESS): return "stress"
    return "baseline"

eth["class"] = eth.apply(classify_row, axis=1)

def get_label(ts):
    rows = eth[(eth["start_dt"] <= ts) & (eth["end_dt"] >= ts)]
    if len(rows)==0: return "baseline"
    if (rows["class"]=="feeding").any(): return "feeding"
    if (rows["class"]=="stress").any(): return "stress"
    return "baseline"

label_map = {"baseline":0, "feeding":1, "stress":2}

# ============================================================
# 5. AUDIO SEGMENTATION + FEATURE EXTRACTION
# ============================================================

def load_audio(path):
    y, sr = sf.read(path)
    if y.ndim>1: y = y.mean(axis=1)
    if sr != SAMPLE_RATE:
        y = librosa.resample(y, sr, SAMPLE_RATE)
    return y

def extract_datetime_from_filename(fname):
    # Expect pattern like: B1_2025-12-22_10-00-00.wav
    base = os.path.basename(fname).split(".")[0]
    parts = base.split("_")
    for p in parts:
        try:
            return datetime.strptime(p, "%Y-%m-%d-%H-%M-%S")
        except:
            pass
    raise ValueError("Filename does not contain datetime:", fname)

def seg_to_logmel(seg):
    S = librosa.feature.melspectrogram(
        y=seg, sr=SAMPLE_RATE, n_fft=1024, hop_length=256,
        n_mels=N_MELS, fmin=150, fmax=12000
    )
    return librosa.power_to_db(S, ref=np.max).astype(np.float32)

def seg_to_mfcc(seg):
    mf = librosa.feature.mfcc(y=seg, sr=SAMPLE_RATE, n_mfcc=N_MFCC)
    return np.concatenate([mf.mean(axis=1), mf.std(axis=1)]).astype(np.float32)

logmels = []
mfccs = []
labels = []

for wav in local_wavs:
    y = load_audio(wav)
    start_dt = extract_datetime_from_filename(wav)

    win = int(WIN_SEC * SAMPLE_RATE)
    hop = int(HOP_SEC * SAMPLE_RATE)

    for i in range(0, len(y)-win, hop):
        seg = y[i:i+win]
        ts = start_dt + timedelta(seconds=i/SAMPLE_RATE)
        lab = get_label(ts)

        logmels.append(seg_to_logmel(seg))
        mfccs.append(seg_to_mfcc(seg))
        labels.append(label_map[lab])

print("Segments:", len(labels))

# ============================================================
# 6. TRADITIONAL ML MODELS
# ============================================================

X = np.stack(mfccs)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2)

rf = RandomForestClassifier(n_estimators=300)
rf.fit(X_train, y_train)
print("\n=== RandomForest ===")
print(classification_report(y_test, rf.predict(X_test)))

svm = SVC(kernel="rbf", C=10)
svm.fit(X_train, y_train)
print("\n=== SVM ===")
print(classification_report(y_test, svm.predict(X_test)))

# ============================================================
# 7. DEEP LEARNING (CNN)
# ============================================================

# Pad mel spectrograms to same width
max_T = max(m.shape[1] for m in logmels)
def pad(m):
    if m.shape[1] == max_T: return m
    return np.pad(m, ((0,0),(0,max_T-m.shape[1])), mode="constant")

logmels = [pad(m) for m in logmels]
Xmel = np.stack(logmels)

X_train, X_test, y_train, y_test = train_test_split(Xmel, y, stratify=y, test_size=0.2)

class MelDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return torch.tensor(self.X[i]).unsqueeze(0), torch.tensor(self.y[i])

train_ds = MelDataset(X_train, y_train)
test_ds  = MelDataset(X_test, y_test)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=32)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        self.fc = nn.Linear(64,3)
    def forward(self,x):
        x = self.net(x)
        return self.fc(x.view(x.size(0),-1))

model = CNN().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()

model.eval()
preds = []
true = []
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(DEVICE)
        p = model(xb).argmax(1).cpu().numpy()
        preds.extend(p)
        true.extend(yb.numpy())

print("\n=== CNN ===")
print(classification_report(true, preds))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 607.0/607.0 kB 3.5 MB/s  0:00:00

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
